In [1]:
from pathlib import Path
import os
import time
import random
import threading
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from data_helpers import encode_sex_to_01, filter_band, scale_support_table

#### Create saving folder

In [2]:
DATADIR = Path("./data_processed")
DATADIR.mkdir(parents=True, exist_ok=True)

#### Load and inspect data

In [4]:
DATA_PATH = Path("./synthetic_data.csv")  # adjust if needed
df = pd.read_csv(DATA_PATH)

# Clean column names
df.columns = (
    df.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

# Ensure required targets exist
#for col in ["age", "MetSCORE", "sex"]:
#    if col not in df.columns:
#        raise KeyError(f"Column '{col}' not found in dataset.")

In [5]:
df

,y_hgb_gdl,y_hct_pct,y_rbc_10^12_per_L,age_years,sex_male,bmi,iron_ugdl,ferritin_ngml,vitamin_d_ngml,folate_ngml,...,creatinine_mgdl,egfr_ml_min,sbp_mmHg,dbp_mmHg,heart_rate_bpm,wbc_10^9_per_L,plt_10^9_per_L,smoker,alcohol_units_per_week,physical_activity_level
0,19.000000,60.000000,6.291353,50,0,24.957576,89.444958,87.255755,34.140913,7.299666,...,1.008780,115.719490,109.603159,72.513854,60.155741,8.502579,243.649304,0,0.000000,2
1,14.359825,48.581512,4.237057,89,0,28.910742,116.562650,42.969526,17.475065,8.127943,...,0.825111,88.029725,105.989341,70.153647,63.153129,7.126684,304.445801,0,11.004784,0
2,13.275444,43.689141,3.903371,89,0,27.056523,56.967295,105.315453,13.556992,6.132826,...,0.997450,86.722029,109.887601,76.572083,67.114741,5.767614,290.449026,1,13.774135,2
3,17.288703,58.641986,5.242943,45,1,30.790172,126.265838,54.219943,26.024412,11.276233,...,1.131734,104.982605,110.979582,74.956105,66.440142,8.940245,331.993325,0,15.657052,0
4,11.057588,36.027439,3.233092,86,1,26.566654,38.904217,76.754523,27.544637,4.547614,...,0.748641,95.014086,136.722275,66.985851,63.148100,8.676900,168.801463,1,22.441643,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29995,15.517019,47.790869,4.611092,68,0,22.940271,118.633284,101.271217,24.823961,5.055702,...,0.758324,106.513944,123.657584,58.593060,62.617462,8.347876,80.000000,0,5.507728,2
29996,15.545615,49.598858,4.459959,30,0,29.090984,74.373798,100.959141,31.439794,8.701332,...,1.161078,122.528003,116.896250,80.842703,57.871766,4.555815,265.441838,0,10.019572,1
29997,18.785686,57.335507,5.522742,72,0,23.157489,114.366288,224.117059,24.374774,6.094330,...,1.026121,92.978461,124.690945,66.279360,75.352123,7.954074,323.863601,0,2.961110,2
29998,19.000000,60.000000,6.041488,88,1,26.708900,114.650159,48.405161,22.265206,10.059077,...,1.003265,77.670446,141.381696,75.705552,95.352669,8.317710,304.177157,0,0.000000,1


#### Encode sex as categorical

#### Drop unused columns

In [6]:
print(df.columns)

Index(['y_hgb_gdl', 'y_hct_pct', 'y_rbc_10^12_per_L', 'age_years', 'sex_male',
       'bmi', 'iron_ugdl', 'ferritin_ngml', 'vitamin_d_ngml', 'folate_ngml',
       'vitamin_b12_pgml', 'crp_mgL', 'albumin_gdl', 'creatinine_mgdl',
       'egfr_ml_min', 'sbp_mmHg', 'dbp_mmHg', 'heart_rate_bpm',
       'wbc_10^9_per_L', 'plt_10^9_per_L', 'smoker', 'alcohol_units_per_week',
       'physical_activity_level'],
      dtype='object')


In [7]:
# Outputs (numeric) used in the working dataframe
# OUTPUT_COLS_NUM = ["age", "age_bin", "MetSCORE", "sex_num"]
OUTPUT_COLS_NUM = ["age_years", "bmi", "sex_male"]

# Columns to always drop from inputs
# DROP_ALWAYS = {"Unnamed: 0", "index", "ID", "Id", "patient_id", "PatientID"}
DROP_ALWAYS = {}

# Inputs = numeric columns excluding outputs
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
INPUT_COLS = [c for c in numeric_cols if c not in OUTPUT_COLS_NUM and c not in DROP_ALWAYS]

print("Number of INPUT_COLS:", len(INPUT_COLS))
print("Example INPUT_COLS:", INPUT_COLS[:10])

df_work = df[INPUT_COLS + OUTPUT_COLS_NUM].dropna().reset_index(drop=True)
print("After dropna:", df_work.shape)

Number of INPUT_COLS: 20
Example INPUT_COLS: ['y_hgb_gdl', 'y_hct_pct', 'y_rbc_10^12_per_L', 'iron_ugdl', 'ferritin_ngml', 'vitamin_d_ngml', 'folate_ngml', 'vitamin_b12_pgml', 'crp_mgL', 'albumin_gdl']
After dropna: (30000, 23)


#### Train - validation - test split --> dataframes

In [8]:
SEED = 1234

train_df, temp_df = train_test_split(df_work, test_size=0.2, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, shuffle=True)

#### Correlation matrices (train split only)

In [9]:
X_corr = train_df[INPUT_COLS]
corr_inputs = X_corr.corr(method="pearson")
kappa = np.linalg.cond(corr_inputs.values)
print(f"Conditioning before pruning: {kappa}")

# corr_inputs_path = DATADIR / "dataframes/corr_inputs_pearson.csv"
# corr_inputs.to_csv(corr_inputs_path, float_format="%.6f")

Conditioning before pruning: 230.6456670659863


In [10]:
# %% Ranking of highly correlated INPUT pairs by bands -> save .csv
C = corr_inputs.copy()
cols = C.columns.tolist()

pairs = []
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = C.iat[i, j]
        if pd.isna(r):
            continue
        pairs.append((cols[i], cols[j], float(r), abs(float(r))))

pairs_df = pd.DataFrame(pairs, columns=["feature_1", "feature_2", "corr", "abs_corr"])
pairs_df = pairs_df.sort_values("abs_corr", ascending=False).reset_index(drop=True)

band_07_08 = filter_band(pairs_df, 0.7, 0.8, include_hi=False)
band_08_09 = filter_band(pairs_df, 0.8, 0.9, include_hi=False)
band_09_10 = filter_band(pairs_df, 0.9, 1.0, include_hi=True)

# band_07_08.to_csv(DATADIR / "dataframes/input_pairs_abs_corr_07_08.csv", 
#                   index=False, float_format="%.6f")
# band_08_09.to_csv(DATADIR / "dataframes/input_pairs_abs_corr_08_09.csv", 
#                   index=False, float_format="%.6f")
# band_09_10.to_csv(DATADIR / "dataframes/input_pairs_abs_corr_09_10.csv", 
#                   index=False, float_format="%.6f")
# pairs_df.to_csv(DATADIR / "dataframes/input_pairs_abs_corr_all.csv", 
#                 index=False, float_format="%.6f")

In [11]:
# %% [PRE] input-output correlations + auto-pruning (>0.85) + scale/support diagnostics
# PRE_DIR = DATADIR / "preprocessing"
FIG_PRE = DATADIR / "figs"
TAB_PRE = DATADIR / "tables"
# PRE_DIR.mkdir(parents=True, exist_ok=True)
FIG_PRE.mkdir(parents=True, exist_ok=True)
TAB_PRE.mkdir(parents=True, exist_ok=True)

In [14]:
# --- input-output correlations (Pearson) for age, MetSCORE, sex
Y_targets = train_df[["age_years", "bmi", "sex_male"]]
corr_in_out = train_df[INPUT_COLS].corrwith(Y_targets["age_years"]).to_frame("corr_with_age")
corr_in_out["corr_with_MetSCORE"] = train_df[INPUT_COLS].corrwith(Y_targets["bmi"])
corr_in_out["corr_with_sex"] = train_df[INPUT_COLS].corrwith(Y_targets["sex_male"])
corr_in_out["abs_sum"] = corr_in_out[["corr_with_age", 
                                      "corr_with_MetSCORE", 
                                      "corr_with_sex"]].abs().sum(axis=1)

corr_in_out_path = TAB_PRE / "corr_input_output_pearson.csv"
corr_in_out.sort_values("abs_sum", ascending=False).to_csv(corr_in_out_path, float_format="%.6f")

#### Detect correlated features

In [15]:
# --- automatic pruning by input-input abs corr threshold (>=0.85)
THR_CORR = 0.85
corr_inputs_abs = corr_inputs.abs().copy()
np.fill_diagonal(corr_inputs_abs.values, 0.0)

relevance = corr_in_out["abs_sum"].to_dict()

kept = set(INPUT_COLS)
dropped = []
pairs_considered = []

pairs_sorted = []
cols = corr_inputs_abs.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr_inputs_abs.iat[i, j]
        if pd.isna(r):
            continue
        if r >= THR_CORR:
            pairs_sorted.append((cols[i], cols[j], float(r)))
pairs_sorted.sort(key=lambda t: -t[2])

for f1, f2, r in pairs_sorted:
    if (f1 not in kept) or (f2 not in kept):
        continue

    s1 = float(relevance.get(f1, 0.0))
    s2 = float(relevance.get(f2, 0.0))

    if s1 > s2:
        drop = f2
    elif s2 > s1:
        drop = f1
    else:
        # tie-breaker: drop the more redundant one (higher avg abs corr)
        avg1 = float(corr_inputs_abs.loc[f1, list(kept)].mean())
        avg2 = float(corr_inputs_abs.loc[f2, list(kept)].mean())
        drop = f1 if avg1 >= avg2 else f2

    kept.remove(drop)
    dropped.append(drop)
    pairs_considered.append((f1, f2, r, s1, s2, drop))

kept = sorted(list(kept))
dropped = sorted(list(set(dropped)))

In [16]:
prune_df = pd.DataFrame(
    pairs_considered,
    columns=["feature_1", "feature_2", "abs_corr", "rel_sum_f1", "rel_sum_f2", "dropped_feature"]
)
prune_df_path = TAB_PRE / f"pruning_pairs_thr_{THR_CORR:.2f}.csv"
# prune_df.to_csv(prune_df_path, index=False, float_format="%.6f")
print("Saved:", prune_df_path)

(Path(TAB_PRE / f"kept_features_thr_{THR_CORR:.2f}.txt")).write_text("\n".join(kept))
(Path(TAB_PRE / f"dropped_features_thr_{THR_CORR:.2f}.txt")).write_text("\n".join(dropped))
print(f"Pruning summary @ thr={THR_CORR:.2f}: kept={len(kept)} dropped={len(dropped)}")

Saved: data_processed/tables/pruning_pairs_thr_0.85.csv
Pruning summary @ thr=0.85: kept=18 dropped=2


In [17]:
support_df = scale_support_table(df, INPUT_COLS)
support_path = TAB_PRE / "scale_support_inputs.csv"
support_df.to_csv(support_path, float_format="%.6f")
print("Saved:", support_path)

Saved: data_processed/tables/scale_support_inputs.csv


#### Remove correlated features

In [18]:
# -----------------------------
# Reproducibility / device
# -----------------------------
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Memory/time profiling utilities
# -----------------------------
try:
    import psutil  # recommended for RSS memory
    _HAS_PSUTIL = True
except Exception:
    psutil = None
    _HAS_PSUTIL = False

In [19]:
# -----------------------------
# Use auto-pruned features from PRE chunk
# -----------------------------
INPUT_COLS_PRUNED = kept
print("N INPUT_COLS_PRUNED:", len(INPUT_COLS_PRUNED))

# -----------------------------
# Model dataframe
# -----------------------------
df_model_train = train_df[INPUT_COLS_PRUNED + ["age_years", 
                                               "bmi", 
                                               "sex_male"]].dropna().reset_index(drop=True)

df_model_val = val_df[INPUT_COLS_PRUNED + ["age_years", 
                                           "bmi", 
                                           "sex_male"]].dropna().reset_index(drop=True)

df_model_test = test_df[INPUT_COLS_PRUNED + ["age_years", 
                                             "bmi", 
                                             "sex_male"]].dropna().reset_index(drop=True)

# df_model_train.to_csv(DATADIR / "dataframes/train_df_pruned.csv", float_format="%.6f")
# df_model_val.to_csv(DATADIR / "dataframes/val_df_pruned.csv", float_format="%.6f")
# df_model_test.to_csv(DATADIR / "dataframes/test_df_pruned.csv", float_format="%.6f")

# print("df_model_train shape:", df_model_train.shape)
# print("df_model_val shape:", df_model_val.shape)
# print("df_model_test shape:", df_model_test.shape)

N INPUT_COLS_PRUNED: 18


In [20]:
X_corr_after = train_df[INPUT_COLS_PRUNED]
corr_inputs_after = X_corr_after.corr(method="pearson")
kappa_after = np.linalg.cond(corr_inputs_after.values)
print(f"Conditioning after pruning: {kappa_after}")

Conditioning after pruning: 32.45488760644779


In [21]:
import pickle
    
with open(DATADIR / "dataframes/input_cols_pruned.pkl", "wb") as f:
    pickle.dump(INPUT_COLS_PRUNED, f)

In [22]:
X_train = df_model_train[INPUT_COLS_PRUNED].values.astype(np.float32)
X_val = df_model_val[INPUT_COLS_PRUNED].values.astype(np.float32)
X_test = df_model_test[INPUT_COLS_PRUNED].values.astype(np.float32)

age_train = df_model_train["age_years"].values.astype(np.float32).reshape(-1, 1)             
age_val = df_model_val["age_years"].values.astype(np.float32).reshape(-1, 1)             
age_test = df_model_test["age_years"].values.astype(np.float32).reshape(-1, 1)             

mets_train = df_model_train["bmi"].values.astype(np.float32).reshape(-1, 1)
mets_val = df_model_val["bmi"].values.astype(np.float32).reshape(-1, 1)
mets_test = df_model_test["bmi"].values.astype(np.float32).reshape(-1, 1)

sex_train = df_model_train["sex_male"].values.astype(np.float32).reshape(-1, 1)
sex_val = df_model_val["sex_male"].values.astype(np.float32).reshape(-1, 1)
sex_test = df_model_test["sex_male"].values.astype(np.float32).reshape(-1, 1)

## SAVE
np.savez(DATADIR / "train_data.npz",
         x=X_train, age=age_train, mets=mets_train, sex=sex_train)

np.savez(DATADIR / "val_data.npz",
         x=X_val, age=age_val, mets=mets_val, sex=sex_val)

np.savez(DATADIR / "test_data.npz",
         x=X_test, age=age_test, mets=mets_test, sex=sex_test)

In [23]:
X_train.shape

(24000, 18)

In [24]:
# -----------------------------
# Normalize inputs + age + MetSCORE (sex stays 0/1)
# -----------------------------
x_mean = X_train.mean(axis=0, keepdims=True)
x_std = X_train.std(axis=0, keepdims=True) + 1e-8
X_train_s = (X_train - x_mean) / x_std
X_val_s = (X_val - x_mean) / x_std
X_test_s = (X_test - x_mean) / x_std

age_mean = age_train.mean(axis=0, keepdims=True)
age_std = age_train.std(axis=0, keepdims=True) + 1e-8
age_train_s = (age_train - age_mean) / age_std
age_val_s = (age_val - age_mean) / age_std
age_test_s = (age_test - age_mean) / age_std

mets_mean = mets_train.mean(axis=0, keepdims=True)
mets_std = mets_train.std(axis=0, keepdims=True) + 1e-8
mets_train_s = (mets_train - mets_mean) / mets_std
mets_val_s = (mets_val - mets_mean) / mets_std
mets_test_s = (mets_test - mets_mean) / mets_std

np.savez(DATADIR / "scalers.npz",
    x_mean=x_mean, x_std=x_std,
    age_mean=age_mean, age_std=age_std,
    mets_mean=mets_mean, mets_std=mets_std)

np.savez(DATADIR / "train_data_scaled.npz",
         x=X_train_s, age=age_train_s, mets=mets_train_s, sex=sex_train)

np.savez(DATADIR / "val_data_scaled.npz",
         x=X_val_s, age=age_val_s, mets=mets_val_s, sex=sex_val)

np.savez(DATADIR / "test_data_scaled.npz",
         x=X_test_s, age=age_test_s, mets=mets_test_s, sex=sex_test)